In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src import utils

In [ ]:
from huggingface_hub import HfFolder, login

api_file = "/home/fre.gilad/source/llm-iml/HF_KEY.txt"
hf_token = utils.api_key_from_file(api_file)

HfFolder.save_token(hf_token)
login(token=hf_token)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_train = data.copy()
ds_eval = data.copy()

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=50, shuffle=False)

In [ ]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

In [ ]:
from src.eval.harmbench_evaluator import HarmBenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    HarmBenchEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=60, verbose=True),
        use_context=False,
    ),
    # LlamaGuardEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     model_name="meta-llama/Meta-Llama-Guard-2-8B",
    # ),
    # StrongRejectEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     # binary_thresh=0.5,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    # ),
    TemplateEvaluator(
    )
]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from notebooks.models import print_models

torch.set_float32_matmul_precision("high")  # negligable effect

print_models()
model_name = "meta-llama/Llama-2-7b-chat-hf"

utils.set_seed(42)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="sequential",
    # attn_implementation="flash_attention_2"
    # attn_implementation="sdpa",
)

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from torch import optim
from src.activation_extractor import ActivationExtractor
from src.attacks.optim_attack import OptimAttack
from src.iml_attack import IML_Attack, StopCriteria
from src.adver_model import AdverModel
from src.initialize import Initializer


adv_model = AdverModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20
)

Initializer.normal(adv_model)

internal_attack = OptimAttack(
    adv_model,
    optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
    steps=30,
    mixed_precision=False,
)

iml_attack = IML_Attack(
    adv_model=adv_model,
    internal_attack=internal_attack,
    optim_factory=lambda params: optim.AdamW(params, lr=2e-2),
    evaluators=evaluators,
    eval_freq=0.33,
    pred_kwargs={"max_length": 512},
    mixed_precision=False, # TODO: test with both true and false
)

stop = StopCriteria(
    max_epochs=5,
    max_time=15 * 60,
    patience=3,
)

In [ ]:
adv_model = iml_attack.fit(dl_train, dl_eval, stop_criteria=stop)

In [ ]:
# TODO: make sure the model computation actually runs at the model dtype
adv_model.set_embeddings(iml_attack.best_embeds)
iml_attack.evaluate(adv_model=adv_model, dl_eval=dl_eval, evalers=evaluators)

In [ ]:
adv_model.set_embeddings(iml_attack.best_embeds)
preds = iml_attack.predict(adv_model, dl_eval, max_length=300)
dl_eval.set_column("response", preds)

for i in range(len(preds)):
    print(f" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(f" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(f" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")